# F02-P2 Nature

**Site Characterization: Nature, components 2.1 and 2.2.**

Characterizes the ecological quality of the AOI forest and habitat. Where `F02-P2 General`
answers "where is this site and what has happened to it", Nature answers "how good is what is
left, and does it matter for biodiversity".

> **Not runnable yet.** All analysis logic below is real Python. Only file access is stubbed.

**Status.** Complete as scoped: 2.1 FLII and 2.2 KBA.

**Known limit of that scope.** Both components need forest or a designated site to say anything. 2.1
FLII is a property of forest and returns not applicable without it, and 2.2 KBA returns a negative
sentence on most sites. A degraded grassland or a cropland targeted for planting therefore receives
almost no ecological description from this module, which is the site type where a RESTORE decision
most depends on the starting condition. Related asymmetry: the reference ecosystem has five classes
in the backend, but the only ecosystem quality metric here measures forest, so savanna and grassland
have no equivalent of FLII. Recorded as a scope limit, not as pending work.

## Handoff

Reads nothing from `F02-P2 General` at present. Both Nature components derive their own forest
extent from `forest_mask_2024`, so this notebook can run independently. It writes
`outputs/<aoi_id>__F02-P2-nature.json`.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

from dataclasses import dataclass
from pathlib import Path

import geopandas as gpd
import pandas as pd
import numpy as np
import os
import rasterio
import matplotlib.pyplot as plt

from pyproj import Geod
from rasterio.mask import mask
from rasterio.mask import raster_geometry_mask
from config import *
from common import *

In [2]:
aoi_id = AOI_ID   # AOI is set in config.py (AOI_PATH, AOI_ID); change it there, then restart the kernel

aoi = prepare_aoi(gpd.read_file(AOI_PATH))
print(f"AOI {aoi_id}: {fmt_ha(aoi.area_ha)}")

results: dict[str, ComponentResult] = {}

AOI aoi1: 67,439 ha


---
## 2.1 Forest Landscape Integrity (FLII)

Reports the landscape integrity of the AOI forest as a headline mean score out of 10, with a
High / Medium / Low breakdown in the narrative. Integrity is the degree to which a forest is
still intact, connected, and free of human pressure.

**Data.** `flii_forest_mosaic_SEA_300m.tif` (continuous 0 to 10, on forest) and
`flii_class_mosaic_SEA_300m.tif` (1 = Low, 2 = Medium, 3 = High, on forest). Based on the Forest
Landscape Integrity Index (Grantham et al. 2020, Nat. Commun. 11:5978), reimplemented and
calibrated on the SEA data stack. Landscape scale, native 300 m.

**Calibration warning.** The 0 to 10 values are calibrated on the pooled SEA distribution, so
they are not one to one with the published global FLII product. Present them as the SEA forest
integrity layer, internally consistent within this run. Do not claim absolute global integrity.
Class breaks follow the paper: High at or above 9.6, Low at or below 6.0, Medium in between.

**Decisions locked.**

- FLII is a property of forest, so the summary covers AOI forest area only. Denominator = AOI
  forest area, the same forest used in 1.5 and 1.6.
- Headline is the mean FLII score out of 10, a single big number for the frontend.
- The narrative reports the High, Medium and Low share and names the predominant class.

**Example render (predominantly High).**

> **Forest landscape integrity: 8.8 / 10**
>
> Of the forest in this area, 68% has high landscape integrity, 24% medium, and 8% low. The
> forest is predominantly high integrity, indicating largely intact and well-connected forest
> under low human pressure.

**Downstream use.** FLII is a biodiversity and ecosystem quality proxy feeding Triple Win
Pillar 1, and a pathway signal: high integrity favours PROTECT, low integrity favours RESTORE or
MANAGE. It also underpins the SCeNe high-integrity NbS criteria.

In [3]:
FLII_LOW, FLII_MEDIUM, FLII_HIGH = 1, 2, 3

FLII_GLOSS = {
    FLII_HIGH: "indicating largely intact and well-connected forest under low human pressure",
    FLII_MEDIUM: (
        "indicating moderately modified forest with some fragmentation or human pressure"
    ),
    FLII_LOW: "indicating heavily modified and fragmented forest under high human pressure",
}


def analyze_flii(aoi: AOI) -> ComponentResult:
    """Component 2.1. Forest landscape integrity over the AOI forest."""
    # The FLII rasters are already masked to forest upstream, so their valid extent defines the
    # forest here. forest_mask_2024 is loaded only to catch the "no forest at all" case early,
    # so the message matches 1.5 and 1.6 rather than saying "no FLII data".
    if forest_mask_2024(aoi).is_empty:
        return not_applicable(
            "2.1 Forest Landscape Integrity",
            "No forest is present in this project area, so landscape integrity cannot be "
            "assessed.",
        )

    score = load_raster_clipped(FLII_FOREST_RASTER, aoi, resampling="bilinear")
    classes = load_raster_clipped(FLII_CLASS_RASTER, aoi, resampling="nearest")

    forest_area_ha = classes.valid_area_ha
    if forest_area_ha <= 0 or score.valid_count == 0:
        return not_applicable(
            "2.1 Forest Landscape Integrity",
            "The forest integrity layer does not cover the forest in this project area.",
        )

    mean_flii = float(np.ma.mean(score.values))  # 0 to 10, one decimal on display

    rows = tabulate_classes(classes, FLII_CLASSES, denominator_ha=forest_area_ha)
    by_code = {r.code: r for r in rows}
    dom = dominant(rows)

    narrative = sentences(
        f"Of the forest in this area, {fmt_pct(by_code[FLII_HIGH].pct)} has high landscape "
        f"integrity, {fmt_pct(by_code[FLII_MEDIUM].pct)} medium, and "
        f"{fmt_pct(by_code[FLII_LOW].pct)} low.",
        f"The forest is predominantly {dom.label.lower()} integrity, {FLII_GLOSS[dom.code]}.",
    )

    return ComponentResult(
        component="2.1 Forest Landscape Integrity",
        applicable=True,
        narrative=narrative,
        tables={"integrity": rows},
        rasters={"2.1_flii_score": score, "2.1_flii_class": classes},
        values={
            "mean_flii": mean_flii,  # headline big number
            "dominant_class": dom.code,
            "pct_high": by_code[FLII_HIGH].pct,
            "forest_area_ha": forest_area_ha,
        },
    )


results["2.1"] = analyze_flii(aoi)
show_result(results["2.1"])

[2.1 Forest Landscape Integrity]
  Of the forest in this area, 0% has high landscape integrity, 45% medium, and 55% low. The forest is predominantly low integrity, indicating heavily modified and fragmented forest under high human pressure.
  integrity:


,code,label,area_ha,pct
0,1,Low,22259.439455,55.058366
1,2,Medium,18169.365774,44.941634
2,3,High,0.000000,0.000000


{'mean_flii': 2.782495220954318,
 'dominant_class': 1,
 'pct_high': 0.0,
 'forest_area_ha': 40428.805228613004}

---
## 2.2 Key Biodiversity Areas (KBA)

Reports whether the AOI overlaps Key Biodiversity Areas, with overlap area, share and a
narrative.

**A KBA is not a protected area.** It is a site that contributes significantly to the global
persistence of biodiversity (IUCN KBA Standard 2016). It may or may not be legally protected.
This is a different lens from 1.3 (WDPA, legal status) and the two complement each other. The
narrative emphasises biodiversity importance, not protection.

**Data.** `KBA_polygon.shp`, World Database of Key Biodiversity Areas, BirdLife International
and the KBA Partnership.

**Decisions locked.**

- Mirrors 1.3 mechanics: headline overlap = union of KBA polygons, so overlapping or nested
  sites are not double counted. No sliver threshold, because any KBA overlap is material.
- Denominator = total AOI area. A KBA concerns the whole site, not only its forest.
- The narrative gives the KBA name only, no criteria or type.

**To verify against the actual file.** Name field assumed here: `IntName`.

**Example render.**

> This project area overlaps 210 ha (17%) of a Key Biodiversity Area, Bukit Tigapuluh. Key
> Biodiversity Areas are sites that contribute significantly to the global persistence of
> biodiversity.

**Downstream use.** Feeds Triple Win Pillar 1 and acts as a safeguard and eligibility signal. A
KBA that is not also under WDPA protection (1.3) is a biodiversity important but unprotected
site, which is a strong PROTECT and additionality rationale.

In [4]:
KBA_DEFINITION = (
    "Key Biodiversity Areas are sites that contribute significantly to the global persistence "
    "of biodiversity."
)


@dataclass(frozen=True)
class KbaSite:
    name: str
    area_ha: float


def analyze_kba(aoi: AOI) -> ComponentResult:
    """Component 2.2. Overlap with Key Biodiversity Areas."""
    gdf = load_vector_intersecting(KBA_POLYGON, aoi)

    if gdf.empty:
        return ComponentResult(
            component="2.2 Key Biodiversity Areas",
            applicable=True,  # "no overlap" is a real answer, not a missing one
            narrative="This project area does not overlap any Key Biodiversity Areas.",
            tables={"sites": []},
            values={"kba_ha": 0.0, "kba_pct": 0.0, "kba_site_count": 0},
        )

    kba_ha = union_overlap_ha(aoi, gdf)
    kba_pct = safe_pct(kba_ha, aoi.area_ha)

    areas = per_feature_overlap_ha(aoi, gdf)
    sites = sort_by_area([
        KbaSite(name=str(gdf.iloc[i].get("IntName", "Unnamed KBA")), area_ha=float(areas[i]))
        for i in range(len(gdf))
    ])

    if len(sites) == 1:
        head = (
            f"This project area overlaps {fmt_ha(kba_ha)} ({fmt_pct(kba_pct)}) of a Key "
            f"Biodiversity Area, {sites[0].name}."
        )
    else:
        largest, *rest = sites
        rest_text = oxford_join(f"{s.name} ({fmt_ha(s.area_ha)})" for s in rest)
        head = (
            f"This project area overlaps {fmt_ha(kba_ha)} ({fmt_pct(kba_pct)}) of Key "
            f"Biodiversity Areas, across {len(sites)} sites. The largest is {largest.name} "
            f"({fmt_ha(largest.area_ha)}), followed by {rest_text}."
        )

    return ComponentResult(
        component="2.2 Key Biodiversity Areas",
        applicable=True,
        narrative=sentences(head, KBA_DEFINITION),
        tables={"sites": sites},
        values={"kba_ha": kba_ha, "kba_pct": kba_pct, "kba_site_count": len(sites)},
    )


results["2.2"] = analyze_kba(aoi)
show_result(results["2.2"])

[2.2 Key Biodiversity Areas]
  This project area overlaps 14,317 ha (21%) of a Key Biodiversity Area, Gunung Niut-Poteng. Key Biodiversity Areas are sites that contribute significantly to the global persistence of biodiversity.
  sites:


,name,area_ha
0,Gunung Niut-Poteng,14316.718525


{'kba_ha': 14316.718524669994,
 'kba_pct': 21.22913156918525,
 'kba_site_count': 1}

---
## 2.3 Habitat Area

## AOH is a refinement of IUCN Range

The **AOH approach** refines coarse species range maps into spatially-explicit estimates of available suitable habitat, following the globally standardized methods of *Brooks et al., 2019*

For each species (~6,000 in total), AOH generated through the following sequential filtering steps:

1. Retain only polygons classified as **extant and native** in the IUCN/BirdLife dataset and remove polygons classified as introduced, uncertain, or extinct.

2. Extract habitat preferences from **IUCN Red List species assessment** and mask the range map to include only pixels corresponding to suitable habitat classes based on the crosswalk.

3. Apply species-specific elevational limits using a high-resolution digital elevation model. Where relevant, additional climatic constraints will be applied to refine habitat suitability for habitat types, such as savannas.

---

The resulting raster represents the intersection of **range, habitat, and environmental suitability**. Outputs will be binary (**suitable/unsuitable**) and stored as species-level rasters.

The accuracy and validity of AOH assessed using species occurrence data from the **Global Biodiversity Information Facility (GBIF)**, as well as data collected from three NbS pilot sites. Validation assess the proportion of independent occurrence records that fall within mapped AOH, providing an indication of model performance

---

## Example render

> The selected area contains suitable habitat for **255 species**.

### Species by Class

| Class         | Species Count |
| ------------- | ------------: |
| **Mammal**    |        **82** |
| **Reptile**   |        **80** |
| **Bird**      |        **50** |
| **Amphibian** |        **43** |

### Species Habitat Intersection

| Class     | Species Name                 | IUCN |  Pixels | Area (ha) |  % AOI |
| --------- | ---------------------------- | :--: | ------: | --------: | -----: |
| Amphibian | `Ansonia_leptopus`           |  LC  | 189,522 | 16,931.50 | 20.68% |
| Amphibian | `Chalcorana_parvaccola`      |  LC  | 635,524 | 56,775.55 | 69.34% |
| Amphibian | `Chalcorana_rufipes`         |  LC  | 248,465 | 22,196.84 | 27.11% |
| Amphibian | `Duttaphrynus_melanostictus` |  LC  | 909,974 | 81,294.72 | 99.28% |
| Amphibian | `Fejervarya_cancrivora`      |  LC  |  50,749 |  4,533.76 |  5.54% |
| Amphibian | `Fejervarya_limnocharis`     |  LC  | 846,355 | 75,611.09 | 92.34% |
| Amphibian | `Hylarana_erythraea`         |  LC  | 804,977 | 71,914.46 | 87.83% |
| Amphibian | `Hylarana_nicobariensis`     |  LC  | 802,407 | 71,684.86 | 87.55% |
| Amphibian | `Ingerophrynus_parvus`       |  LC  | 467,141 | 41,732.63 | 50.97% |
| Amphibian | `Kalophrynus_pleurostigma`   |  LC  | 774,827 | 69,220.91 | 84.54% |




In [ ]:

def geometry_area_ha(geometry):
    """Calculate geodesic polygon area in hectares."""

    area_m2, _ = GEOD.geometry_area_perimeter(
        geometry
    )

    return abs(area_m2) / 10000


def pixel_area_by_row(transform, height):
    """
    Calculate geodesic pixel area by raster row.

    Required for EPSG:4326 because pixel area changes with latitude.
    """

    pixel_width = abs(transform.a)

    row_areas_ha = np.zeros(height)

    for row_index in range(height):

        north = (
            transform.f
            + row_index * transform.e
        )

        south = (
            north
            + transform.e
        )

        west = transform.c
        east = west + pixel_width

        area_m2, _ = GEOD.polygon_area_perimeter(
            [west, east, east, west],
            [north, north, south, south]
        )

        row_areas_ha[row_index] = (
            abs(area_m2) / 10000
        )

    return row_areas_ha


# =============================================================================
# Main analysis
# =============================================================================

def analyze_biodiversity(
    user_polygon,
    inventory_path,
    biodiversity_root,
    target_dn=1
):

    # -------------------------------------------------------------------------
    # 1. Read AOI
    # -------------------------------------------------------------------------

    aoi = gpd.read_file(
        user_polygon
    )

    if aoi.empty:
        raise ValueError(
            "The AOI contains no features."
        )

    if aoi.crs is None:
        raise ValueError(
            "The AOI has no CRS."
        )

    # GeoParquet inventory is expected in EPSG:4326
    aoi = aoi.to_crs(
        "EPSG:4326"
    )

    valid_geometry = aoi.geometry[
        aoi.geometry.notna()
        & ~aoi.geometry.is_empty
    ]

    if valid_geometry.empty:
        raise ValueError(
            "The AOI contains no valid geometry."
        )

    aoi_geometry = (
        valid_geometry.union_all()
    )

    aoi_area_ha = (
        geometry_area_ha(
            aoi_geometry
        )
    )


    # -------------------------------------------------------------------------
    # 2. Load master GeoParquet
    # -------------------------------------------------------------------------

    inventory = gpd.read_parquet(
        inventory_path
    )

    required_columns = {
        "species",
        "class",
        "raster_path",
        "iucn_status",
        "redlistCategory",
        "geometry"
    }

    missing_columns = (
        required_columns
        - set(inventory.columns)
    )

    if missing_columns:
        raise KeyError(
            f"Missing GeoParquet columns: "
            f"{sorted(missing_columns)}"
        )

    if inventory.crs != aoi.crs:
        inventory = inventory.to_crs(
            aoi.crs
        )


    # -------------------------------------------------------------------------
    # 3. Spatial filter candidate species
    # -------------------------------------------------------------------------

    candidates = inventory[
        inventory.intersects(
            aoi_geometry
        )
    ].copy()


    print(
        f"Candidate species: "
        f"{len(candidates)} / {len(inventory)}"
    )


    # -------------------------------------------------------------------------
    # 4. Analyze candidate rasters
    # -------------------------------------------------------------------------

    results = []


    for index, (_, row) in enumerate(
        candidates.iterrows(),
        start=1
    ):

        taxon_class = row["class"]
        species = row["species"]

        raster_path = (
            biodiversity_root
            / row["raster_path"]
        )


        print(
            f"[{index}/{len(candidates)}] "
            f"{taxon_class} | {species}"
        )


        if not raster_path.exists():

            print(
                f"  Missing raster: "
                f"{raster_path}"
            )

            continue


        try:

            with rasterio.open(
                raster_path
            ) as src:


                # -------------------------------------------------------------
                # AOI geometry in raster CRS
                # -------------------------------------------------------------

                if src.crs != aoi.crs:

                    raster_aoi = (
                        gpd.GeoSeries(
                            [aoi_geometry],
                            crs=aoi.crs
                        )
                        .to_crs(src.crs)
                        .iloc[0]
                    )

                else:

                    raster_aoi = (
                        aoi_geometry
                    )


                # -------------------------------------------------------------
                # 5. Get only AOI raster window
                # -------------------------------------------------------------

                try:

                    outside_mask, transform, window = (
                        raster_geometry_mask(
                            src,
                            [
                                raster_aoi
                                .__geo_interface__
                            ],
                            crop=True,
                            all_touched=False
                        )
                    )

                except ValueError:

                    continue


                # -------------------------------------------------------------
                # 6. Read only AOI portion
                # -------------------------------------------------------------

                data = src.read(
                    1,
                    window=window,
                    masked=True
                )


                valid_pixels = (
                    ~outside_mask
                    & ~np.ma.getmaskarray(
                        data
                    )
                )


                # -------------------------------------------------------------
                # 7. Detect DN == 1 habitat
                # -------------------------------------------------------------

                presence_mask = (
                    valid_pixels
                    & (
                        data.data
                        == target_dn
                    )
                )


                pixel_count = int(
                    presence_mask.sum()
                )


                # GeoParquet footprint overlaps,
                # but actual DN 1 habitat does not.
                if pixel_count == 0:

                    continue


                # -------------------------------------------------------------
                # 8. Calculate habitat area
                # -------------------------------------------------------------

                if src.crs.to_epsg() == 4326:

                    row_area_ha = (
                        pixel_area_by_row(
                            transform,
                            data.shape[0]
                        )
                    )


                    pixels_per_row = (
                        presence_mask.sum(
                            axis=1
                        )
                    )


                    habitat_area_ha = float(
                        np.sum(
                            pixels_per_row
                            * row_area_ha
                        )
                    )


                else:

                    # Valid when projected CRS units are metres.
                    pixel_area_ha = (
                        abs(
                            transform.a
                            * transform.e
                        )
                        / 10000
                    )

                    habitat_area_ha = (
                        pixel_count
                        * pixel_area_ha
                    )


                # -------------------------------------------------------------
                # 9. Calculate percentage of AOI
                # -------------------------------------------------------------

                percentage_aoi = (
                    habitat_area_ha
                    / aoi_area_ha
                    * 100
                    if aoi_area_ha > 0
                    else 0
                )


                # -------------------------------------------------------------
                # 10. Store species result
                # -------------------------------------------------------------

                results.append({

                    "species":
                        species,

                    "class":
                        taxon_class,

                    "iucn_status":
                        row["iucn_status"],

                    "redlistCategory":
                        row["redlistCategory"],

                    "pixel_count":
                        pixel_count,

                    "habitat_area_ha":
                        round(
                            habitat_area_ha,
                            2
                        ),

                    "percentage_aoi":
                        round(
                            percentage_aoi,
                            2
                        )
                })


        except rasterio.errors.RasterioIOError as error:

            print(
                f"  Failed: {error}"
            )


    # -------------------------------------------------------------------------
    # 11. Create result table
    # -------------------------------------------------------------------------

    result_table = pd.DataFrame(
        results
    )


    if not result_table.empty:

        result_table = (
            result_table
            .sort_values(
                [
                    "class",
                    "species"
                ]
            )
            .reset_index(
                drop=True
            )
        )


    # -------------------------------------------------------------------------
    # 12. Summary by class
    # -------------------------------------------------------------------------

    if result_table.empty:

        class_summary = {}

    else:

        class_summary = (
            result_table[
                "class"
            ]
            .value_counts()
            .to_dict()
        )


    # -------------------------------------------------------------------------
    # 13. Summary by IUCN status
    # -------------------------------------------------------------------------

    if result_table.empty:

        iucn_summary = {}

    else:

        iucn_summary = (
            result_table[
                "iucn_status"
            ]
            .value_counts()
            .to_dict()
        )


    # -------------------------------------------------------------------------
    # Return
    # -------------------------------------------------------------------------

    return {

        "aoi_area_ha":
            round(
                aoi_area_ha,
                2
            ),

        "inventory_species_count":
            len(inventory),

        "candidate_species_count":
            len(candidates),

        "species_count":
            len(result_table),

        "class_summary":
            class_summary,

        "iucn_summary":
            iucn_summary,

        "species":
            result_table
    }


# =============================================================================
# Run analysis
# =============================================================================

result = analyze_biodiversity(
    user_polygon=USER_POLYGON,
    inventory_path=INVENTORY_PATH,
    biodiversity_root=BIODIVERSITY_ROOT,
    target_dn=TARGET_DN
)


# =============================================================================
# Display summary
# =============================================================================

print()
print(
    f"AOI area: "
    f"{result['aoi_area_ha']:,.2f} ha"
)

print(
    f"The selected area contains suitable habitat for "
    f"{result['species_count']} species."
)


# =============================================================================
# Species summary by class
# =============================================================================

print()
print("Species by Class:")
print("-" * 40)

for taxon_class, count in (
    result["class_summary"]
    .items()
):

    print(
        f"{taxon_class:<20}"
        f"{count:>5}"
    )


# =============================================================================
# IUCN summary
# =============================================================================

print()
print("Species by IUCN Status:")
print("-" * 40)

for status, count in (
    result["iucn_summary"]
    .items()
):

    print(
        f"{status:<20}"
        f"{count:>5}"
    )


# =============================================================================
# Detailed species table
# =============================================================================

if not result["species"].empty:

    print()
    print(
        "Species Habitat Intersection:"
    )

    print("-" * 125)

    print(
        f"{'Class':<15}"
        f"{'Species Name':<40}"
        f"{'IUCN':<8}"
        f"{'Pixels':>12}"
        f"{'Area (ha)':>18}"
        f"{'% AOI':>14}"
    )

    print("-" * 125)


    for _, row in (
        result["species"]
        .iterrows()
    ):

        print(
            f"{str(row['class']):<15}"
            f"{str(row['species']):<40}"
            f"{str(row['iucn_status']):<8}"
            f"{int(row['pixel_count']):>12,}"
            f"{float(row['habitat_area_ha']):>18,.2f}"
            f"{float(row['percentage_aoi']):>13.2f}%"
        )


# =============================================================================
# JSON-ready backend response
# =============================================================================

backend_response = {

    "aoi_area_ha":
        result["aoi_area_ha"],

    "species_count":
        result["species_count"],

    "species_by_class":
        result["class_summary"],

    "iucn_summary":
        result["iucn_summary"],

    "species":
        result["species"]
        .to_dict(
            orient="records"
        )
}


print()
print(
    backend_response
)

---
## 2.5 Key Species Presence

Reports the species occurence inside the AOI, with overlap point, share the quantitative data

The Key Species Presence is occurence of keystone or flagship species from different region accross Southeast-Asia. The data comes from GBIF (Global Biodiversity Information Facility) by compiling sources from Natural History Museums, Citizen Science Networks, Academic & Research Institutions, Government & Conservation Agencies. Following the data standards under The Darwin Core Standard (DwC), Ecological Metadata Language, and The Biological Collection Access Service.

---

The resulting shapefile represents the intersection of **species, individualCount, eventDate and basisofRecords**. Main outputs will be the total occurence (**Species Name and Record Count**)

---

## Example render

> The selected area recorded observations of **2** species.

| Occurrence summary    | Value |
| --------------------- | ----: |
| **Unique species**    | **2** |
| **Total records**     |     4 |
| **Total individuals** |     6 |

### Species Occurrence

| Species Name               | Total Occurrence | Record Count | Latest Encounter | Basis of Record   |
| -------------------------- | ---------------: | -----------: | ---------------- | ----------------- |
| `Buceros rhinoceros`       |            **6** |            3 | 23 February 2017 | HUMAN_OBSERVATION |
| `Symphalangus syndactylus` |            **0** |            1 | 03 November 2014 | HUMAN_OBSERVATION |



In [ ]:
# ---- Species occurrence points -----------------------------------------------
species_points_path = r"\\OPENMEDIAVAULT\geospatial\NBSTOOLV3\BIODIVERSITY\GBIF\key_species.shp"

# ---- User polygon ------------------------------------------------------------
user_polygon_path = r"Z:\NbS_Tools\Dummy\AOI_4326.shp"

# ---- Output ------------------------------------------------------------------
output_excel_path = (
    r"\\OPENMEDIAVAULT\geospatial\NBS\AoH\temp_pilot\species_occurrence_summary.xlsx"
)

# ---- Desired attribute columns (Darwin Core names) ---------------------------
# The resolver matches these against the (possibly truncated) shapefile fields.
SPECIES_COLUMN = "species"
COUNT_COLUMN = "individualCount"
DATE_COLUMN = "eventDate"
BASIS_COLUMN = "basisOfRecord"

# Spatial predicate: "intersects" includes points on the boundary,
# "within" is strictly interior.
SPATIAL_PREDICATE = "intersects"


# =============================================================================
# Helpers
# =============================================================================

def resolve_column(gdf, desired):
    """Find the actual column matching a desired name, allowing for
    shapefile 10-char truncation and case differences."""
    columns = list(gdf.columns)

    # 1. Exact match.
    if desired in columns:
        return desired

    lower_map = {col.lower(): col for col in columns}

    # 2. Case-insensitive exact match.
    if desired.lower() in lower_map:
        return lower_map[desired.lower()]

    # 3. Truncation: the shapefile field is a prefix of the desired name.
    #    Pick the longest such prefix match.
    candidates = [
        col for col in columns
        if desired.lower().startswith(col.lower()) and len(col) >= 4
    ]
    if candidates:
        return max(candidates, key=len)

    raise KeyError(
        f"Could not find a column for '{desired}'. "
        f"Available columns: {columns}"
    )


def most_frequent(series):
    modes = series.dropna().mode()
    return modes.iat[0] if not modes.empty else None


# =============================================================================
# Load inputs
# =============================================================================

points = gpd.read_file(species_points_path)
if points.empty:
    raise ValueError("The species occurrence layer contains no features.")

polygon = gpd.read_file(user_polygon_path)
if polygon.empty:
    raise ValueError("The user polygon contains no features.")

print(f"Loaded {len(points)} occurrence points.")

# Resolve the attribute columns against the actual (truncated) field names.
species_col = resolve_column(points, SPECIES_COLUMN)
count_col = resolve_column(points, COUNT_COLUMN)
date_col = resolve_column(points, DATE_COLUMN)
basis_col = resolve_column(points, BASIS_COLUMN)

print("Resolved columns:")
print(f"  species        -> {species_col}")
print(f"  individualCount-> {count_col}")
print(f"  eventDate      -> {date_col}")
print(f"  basisOfRecord  -> {basis_col}")


# =============================================================================
# Steps 1-2 - Keep points inside the polygon
# =============================================================================

# Align CRS: reproject the polygon to the points CRS.
if points.crs is None:
    raise ValueError("The occurrence layer has no CRS.")

if polygon.crs is None:
    print("Warning: polygon has no CRS; assuming it matches the points layer.")
    polygon = polygon.set_crs(points.crs)
elif polygon.crs != points.crs:
    polygon = polygon.to_crs(points.crs)

# Dissolve the polygon to a single geometry so a point can't match more than
# one polygon feature (which would double-count it).
try:
    merged_geometry = polygon.geometry.union_all()
except AttributeError:  # older geopandas
    merged_geometry = polygon.geometry.unary_union

polygon_single = gpd.GeoDataFrame(geometry=[merged_geometry], crs=polygon.crs)

clipped_points = gpd.sjoin(
    points,
    polygon_single,
    predicate=SPATIAL_PREDICATE,
    how="inner",
).drop(columns="index_right")

print(f"Occurrence points inside the polygon: {len(clipped_points)}")

if clipped_points.empty:
    raise SystemExit("No occurrence points fall inside the polygon.")


# =============================================================================
# Step 3-4 - Clean, group by species, aggregate
# =============================================================================

# individualCount -> numeric (nulls become NaN and are ignored by sum).
clipped_points[count_col] = pd.to_numeric(
    clipped_points[count_col], errors="coerce"
)

# eventDate -> datetime so "max" is chronological, not lexical.
clipped_points[date_col] = pd.to_datetime(
    clipped_points[date_col], errors="coerce"
)

# Records with no species name cannot be grouped meaningfully.
clipped_points = clipped_points.dropna(subset=[species_col])

result = (
    clipped_points
    .groupby(species_col)
    .agg(
        total_occurrence=(count_col, "sum"),      # summed individualCount
        record_count=(species_col, "count"),      # number of occurrence rows
        latest_encounter=(date_col, "max"),
        basis_of_record=(basis_col, most_frequent),
    )
    .reset_index()
    .rename(columns={species_col: "species"})
)


# =============================================================================
# Step 5 - Total unique species
# =============================================================================

total_unique_species = result["species"].nunique()


# =============================================================================
# Format output
# =============================================================================

result["total_occurrence"] = result["total_occurrence"].round().astype("Int64")
result["record_count"] = result["record_count"].astype("Int64")

result["latest_encounter"] = result["latest_encounter"].apply(
    lambda value: value.strftime("%d %B %Y") if pd.notna(value) else None
)

result = result.sort_values(
    "total_occurrence", ascending=False
).reset_index(drop=True)


# =============================================================================
# Write and display
# =============================================================================

output_path = Path(output_excel_path)
output_path.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    result.to_excel(writer, sheet_name="species_summary", index=False)

    worksheet = writer.sheets["species_summary"]
    for column_cells in worksheet.columns:
        longest = max(
            (len(str(cell.value)) for cell in column_cells if cell.value is not None),
            default=0,
        )
        letter = column_cells[0].column_letter
        worksheet.column_dimensions[letter].width = min(longest + 2, 60)

# print()
# print(f"Total unique species: {total_unique_species}")
# print()
# print(result.to_string(index=False))
# print()
# print(f"Summary written to: {output_path}")

# ---- Notebook rendering ------------------------------------------------------
# In Jupyter / ArcGIS notebooks this shows the summary as a formatted table.
try:
    from IPython.display import display

    print()
    print(f"Total unique species: {total_unique_species}")
    display(result)
except ImportError:
    pass

# Leaving `result` as the final expression lets a notebook cell render it too.
result

---
## 2.6 Conservation Significant

NatureMap priority-rank analysis for project AOIs.

Data:  NatureMap_prioritymaps.zip (Jung et al. 2021, Nat Ecol Evol 5:1499-1509;
       Zenodo 10.5281/zenodo.5006332). Ranked priority layers only -- this archive
       contains NO raw carbon or water stock rasters.

Axes are % of AOI land area falling within the global top-X% ranked cells, one
axis per weighting scenario. This is NOT the same quantity as the paper's Fig. 1
triangle (which shows % of species targets met / % of stock conserved).

Resolution: 10 km (100 km2 per pixel), native -- nothing is resampled here.
Suitable for landscape-context statements, not for site-level or per-stratum figures.

> **AOI_A** covers **1,456 km²** and overlaps **23** valid NatureMap priority pixels.

| Area summary           | Value                       |
| ---------------------- | --------------------------: |
| **True polygon area**  |  **1,456 km²** (145,642 ha) |
| **Pixel envelope**     |      2,300 km² (230,000 ha) |
| **Envelope inflation** |                       +58 % |
| **Valid pixels**       |                          23 |

### Priority Rank by Axis

| Axis           | Min Rank | Valid Pixels |
| -------------- | :------: | -----------: |
| `Biodiversity` |  **2.0** |           23 |
| `Carbon`       |  **1.0** |           23 |
| `Water`        |  **2.0** |           23 |



In [ ]:
LAYERS = {
    "Biodiversity": r"Z:\Biodiversity\biodiversity_only_v3.tif",
    "Carbon":       r"Z:\Biodiversity\biodiversity_carbon_v3.tif",
    "Water":        r"Z:\Biodiversity\biodiversity_water_v3.tif",
}
BUDGETS = [10, 30]

AOI_NAME = "AOI_A"
AOI_SHP  = "Z:/NbS_Tools/Dummy/AOI_4326.shp"

OUT = "Z:/NbS_Tools/Dummy/output"
os.makedirs(OUT, exist_ok=True)

EARTH_R_KM = 6371.0088          # mean Earth radius

# ---------------------------------------------------------------- helpers
def pixel_area_km2(transform, shape):
    """True ground area (km2) of every cell in a 4326 raster window.

    Longitude cells are equal width everywhere; latitude bands shrink toward the
    poles. Area of a cell spanning [lat1, lat2] x d_lon degrees:
        R^2 * d_lon_rad * (sin(lat2) - sin(lat1))
    Returns a 2-D array matching `shape` (rows x cols)."""
    d_lon = abs(transform.a)                 # degrees per pixel in x
    d_lat = abs(transform.e)                 # degrees per pixel in y
    n_rows, n_cols = shape

    # latitude of each row's top and bottom edge
    top = transform.f                        # y of the upper-left corner
    row_top = top - np.arange(n_rows) * d_lat
    row_bot = row_top - d_lat
    band = (np.sin(np.radians(row_top)) - np.sin(np.radians(row_bot)))  # per row
    d_lon_rad = np.radians(d_lon)
    row_area = (EARTH_R_KM ** 2) * d_lon_rad * band          # km2 per cell, per row
    return np.repeat(row_area[:, None], n_cols, axis=1)


def clip(path, geoms):
    """Clip raster to geometry. all_touched=True keeps any pixel the polygon
    intersects. Returns (values, area_km2) as 2-D arrays with NaN outside."""
    with rasterio.open(path) as src:
        arr, transform = mask(src, geoms, crop=True, all_touched=True,
                              nodata=src.nodata, filled=True)
        vals = arr[0].astype("float64")
        if src.nodata is not None:
            vals[vals == src.nodata] = np.nan
    area = pixel_area_km2(transform, vals.shape)
    area = np.where(np.isnan(vals), np.nan, area)   # mask area outside polygon too
    return vals, area


def analyse(name, shp):
    aoi = gpd.read_file(shp)
    if aoi.crs is None:
        raise ValueError(f"{shp} has no CRS defined. If lon/lat: aoi = aoi.set_crs(4326)")
    aoi = aoi.to_crs(4326)                    # rasters are 4326; match them
    geoms = [aoi.union_all().__geo_interface__]

    # true polygon area: reproject a copy to equal-area purely for the area number
    true_km2 = aoi.to_crs("ESRI:54009").area.sum() / 1e6

    # envelope + per-pixel areas come from the first layer (all share the grid)
    ref_vals, ref_area = clip(next(iter(LAYERS.values())), geoms)
    envelope = np.nansum(ref_area)
    n_px     = int(np.sum(~np.isnan(ref_vals)))

    print(f"\n=== {name} ===")
    print(f"  true polygon area : {true_km2:,.0f} km2 ({true_km2 * 100:,.0f} ha)")
    print(f"  pixel envelope    : {envelope:,.0f} km2 ({envelope * 100:,.0f} ha)")
    if true_km2 > 0:
        print(f"  envelope inflation: {100 * envelope / true_km2 - 100:+.0f}%")
    print(f"  valid pixels      : {n_px}")
    if n_px == 0:
        raise ValueError(f"{name}: no overlap with raster grid -- check CRS/extent.")
    if n_px < 20:
        print(f"  ** WARNING: {n_px} pixels. Percentages are unstable at this count **")

    rows = []
    for label, path in LAYERS.items():
        rank, area = clip(path, geoms)
        valid = int(np.sum(~np.isnan(rank)))
        rmin  = np.nanmin(rank) if valid else np.nan
        print(f"  {label:14s} min rank {rmin:6.1f}   valid px {valid}")

        for b in BUDGETS:
            sel = rank <= b
            km2 = np.nansum(area[sel])
            rows.append({
                "aoi": name, "scenario": label, "budget_pct": b,
                "area_km2": km2, "area_ha": km2 * 100,
                "pct_of_envelope": 100 * km2 / envelope,
                "min_rank": rmin, "valid_px": valid,
                "true_km2": true_km2, "envelope_km2": envelope,
            })
    return pd.DataFrame(rows)


def radar(df, name, stem):
    axes   = list(LAYERS.keys())
    angles = np.linspace(0, 2 * np.pi, len(axes), endpoint=False)
    angles = np.concatenate([angles, angles[:1]])
    colors = {10: "#8B1A1A", 30: "#C9A227"}

    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"projection": "polar"})
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)

    n_px = int(df[df.aoi == name]["valid_px"].max())

    for b in BUDGETS:
        sub  = df[(df.aoi == name) & (df.budget_pct == b)].set_index("scenario")
        vals = [sub.loc[a, "pct_of_envelope"] for a in axes]
        if sum(v > 0 for v in vals) < 3:
            print(f"  note: {name} {b}% has zero-value axes -- polygon degenerates")
        vals += vals[:1]
        ax.plot(angles, vals, color=colors[b], lw=2.5, marker="o", label=f"{b}%")
        ax.fill(angles, vals, color=colors[b], alpha=0.08)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(axes, fontsize=12)
    ax.set_ylim(0, 100)
    ax.set_yticks([0, 50, 100])
    ax.set_yticklabels(["0%", "50%", "100%"], fontsize=9)
    ax.set_title(f"{name} — AOI area within global priority ranks\n"
                 f"NatureMap 10 km, EPSG:4326 ({n_px} pixels)", fontsize=11, pad=24)
    ax.legend(title="Global top-X% rank", loc="upper right",
              bbox_to_anchor=(1.32, 1.10), fontsize=9)

    plt.savefig(stem + ".svg", bbox_inches="tight")
    plt.savefig(stem + ".png", dpi=300, bbox_inches="tight")
    plt.show()          # renders inline in the notebook


# ---------------------------------------------------------------- run
results = analyse(AOI_NAME, AOI_SHP)
results.to_csv(f"{OUT}/aoi_priority_summary.csv", index=False)
radar(results, AOI_NAME, f"{OUT}/radar_{AOI_NAME}")

print("\n--- summary (% of pixel envelope) ---")
print(results.pivot_table(index="scenario", columns="budget_pct",
                          values="pct_of_envelope").round(1))

---
## Save

Each section above already ran and displayed itself. This cell writes them to one combined JSON.

In [ ]:
path = save_results(results, aoi, aoi_id, STAGE_NATURE)
print(f"Saved {path}")